In [1]:
import sys, os
import copy
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
import torch
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/lunli/Documents/QuantLib/Untitled
Fixed Income Library is loaded.


In [2]:
### utility functions
#
# New Interpolator1D API:
#   interp.interpolate(x, calc_grad=False) -> torch.Tensor
#   interp.integrate(x_s, x_e, calc_grad=False) -> torch.Tensor
# `x` can be a plain scalar or an array-like; no manual array wrapping needed.
# To get sensitivities, call with calc_grad=True, call .backward() on the
# (summed) result *outside* the interpolator, then read interp.values_.grad.
# Interpolator2D keeps the older calc_grad/convert_to_numpy + gradient_wrt_ordinate API.

def to_np(t):
    if isinstance(t, torch.Tensor):
        return t.detach().numpy()
    return np.asarray(t)


def analytic_grad_wrt_ordinate_1d(
    x,
    axis1_1d : List,
    values_1d : List,
    interp_method_1d : str,
    extrap_method_1d : str):
    """grad of sum_i interp(x_i) wrt values, via backward() called outside interpolate()."""
    interp = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    out = interp.interpolate(x, calc_grad=True)
    out_sum = out if out.dim() == 0 else out.sum()
    out_sum.backward()
    return interp.values_.grad.detach().numpy()


def analytic_grad_of_integrated_value_wrt_ordinate_1d(
    x_s,
    x_e,
    axis1_1d : List,
    values_1d : List,
    interp_method_1d : str,
    extrap_method_1d : str):
    """grad of sum_i integrate(x_s_i, x_e_i) wrt values, via backward() called outside integrate()."""
    interp = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    out = interp.integrate(x_s, x_e, calc_grad=True)
    out_sum = out if out.dim() == 0 else out.sum()
    out_sum.backward()
    return interp.values_.grad.detach().numpy()


def bump_reval_interpolator(
    x : float,
    axis1_1d : List,
    values_1d : List,
    interp_method_1d : str,
    extrap_method_1d : str,
    bump_size : Optional[float]=1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    b_value = to_np(base_interpolator.interpolate(x))

    grad = []
    for i in range(len(values_1d)):
        values_1d[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
        bumped_value = to_np(this_interp.interpolate(x))
        grad.append(np.sum((bumped_value - b_value) / bump_size))
        values_1d[i] -= bump_size

    return np.array(grad)

def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1_1d : List,
    values_1d : List,
    interp_method_1d : str,
    extrap_method_1d : str,
    bump_size : Optional[float]=1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    b_value = to_np(base_interpolator.integrate(x_s, x_e))

    grad = []
    for i in range(len(values_1d)):
        values_1d[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
        bumped_value = to_np(this_interp.integrate(x_s, x_e))
        grad.append(np.sum((bumped_value - b_value) / bump_size))
        values_1d[i] -= bump_size

    return np.array(grad)

## Test 1D Interpolator -- Interpolate and Sensitivities

In [3]:
axis1_1d = [1, 3, 5, 7]
values_1d = [3, 4, 5, 6]
interp_method_1d = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method_1d = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1_1d, values_1d, interp_method_1d, extrap_method_1d)

# (x, expected, description) -- regular cases + corner cases (knots, extrapolation)
test_cases = [
    (1,    4,  'left knot [1,3)'),
    (1.5,  4,  'interior [1,3)'),
    (3,    5,  'middle knot [3,5)'),
    (5,    6,  'knot [5,7)'),
    (5.5,  6,  'interior [5,7)'),
    (7,    6,  'right boundary (flat extrap)'),
    (0.5,  3,  'left extrapolation'),
    (-99,  3,  'far left extrapolation'),
    (6.5,  6,  'interior [5,7)'),
    (100,  6,  'far right extrapolation'),
]
tol = 1e-12
print(f"{'Description':<30}  {'x':>8}  {'Result':>10}  {'Expected':>10}  Status")
print('-' * 72)
for x, expected, desc in test_cases:
    result = float(interp_1d.interpolate(x))
    status = 'PASS' if abs(result - expected) < tol else 'FAIL'
    print(f"{desc:<30}  {x:>8}  {result:>10.4f}  {expected:>10.4f}  [{status}]")

Description                            x      Result    Expected  Status
------------------------------------------------------------------------
left knot [1,3)                        1      4.0000      4.0000  [PASS]
interior [1,3)                       1.5      4.0000      4.0000  [PASS]
middle knot [3,5)                      3      5.0000      5.0000  [PASS]
knot [5,7)                             5      6.0000      6.0000  [PASS]
interior [5,7)                       5.5      6.0000      6.0000  [PASS]
right boundary (flat extrap)           7      6.0000      6.0000  [PASS]
left extrapolation                   0.5      3.0000      3.0000  [PASS]
far left extrapolation               -99      3.0000      3.0000  [PASS]
interior [5,7)                       6.5      6.0000      6.0000  [PASS]
far right extrapolation              100      6.0000      6.0000  [PASS]


In [4]:
### 1D interpolate: scalar input does not need to be wrapped in an array
x_scalar = 2.5
out_scalar = interp_1d.interpolate(x_scalar)
print('scalar input -> output:', out_scalar, ' ndim:', out_scalar.dim())
assert out_scalar.dim() == 0, "scalar input should produce a 0-dim tensor, no array wrapping required"
print('[PASS] scalar passthrough works without array conversion')

scalar input -> output: tensor(4., dtype=torch.float64)  ndim: 0
[PASS] scalar passthrough works without array conversion


In [5]:
### 1D interpolate gradient: analytic (backward called outside interpolate()) vs bump-reval
tol = 1e-6
test_xs = [
    (1,    'left knot'),
    (1.5,  'interior [1,3)'),
    (3,    'middle knot'),
    (5,    'knot [5,7)'),
    (7,    'right boundary'),
    (0.5,  'left extrapolation'),
    (-99,  'far left extrapolation'),
    (6.5,  'interior [5,7)'),
    (100,  'far right extrapolation'),
]
print(f"{'Description':<30}  {'x':>8}  {'max|diff|':>12}  Status")
print('-' * 65)
for x, desc in test_xs:
    grad_analytic = analytic_grad_wrt_ordinate_1d(x, axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    grad_br = bump_reval_interpolator(x, axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    max_diff = np.abs(grad_analytic - grad_br).max()
    status = 'PASS' if max_diff < tol else 'FAIL'
    print(f"{desc:<30}  {x:>8}  {max_diff:>12.2e}  [{status}]")

Description                            x     max|diff|  Status
-----------------------------------------------------------------
left knot                              1      4.44e-12  [PASS]
interior [1,3)                       1.5      2.11e-12  [PASS]
middle knot                            3      2.33e-12  [PASS]
knot [5,7)                             5      2.33e-12  [PASS]
right boundary                         7      2.33e-12  [PASS]
left extrapolation                   0.5      2.11e-12  [PASS]
far left extrapolation               -99      2.11e-12  [PASS]
interior [5,7)                       6.5      2.33e-12  [PASS]
far right extrapolation              100      2.33e-12  [PASS]


## Test 1D Interpolator -- Integral and Sensitivities

In [6]:
### 1D integration
# (x_s, x_e, expected, description) -- regular spans + corner cases (zero-width, reversed wings)
test_cases = [
    (0.5,  0.9,   1.2,  'both in left wing'),
    (0.5,  1.2,   2.3,  'left wing to first bucket'),
    (0.5,  3.2,  10.5,  'left wing to middle'),
    (1.5,  5.2,  17.2,  'interior span'),
    (3.5,  7.2,  20.7,  'middle to right wing'),
    (6,    7.2,   7.2,  'last bucket to right wing'),
    (8,   10,    12.0,  'both in right wing'),
    (0.1, 10,    50.7,  'full span'),
    (2,    2,     0.0,  'zero-width interval'),
    (1,    3,     8.0,  'knot-to-knot [1,3)'),
    (1,    7,    30.0,  'full interior [1,7)'),
    (-5,  15,    96.0,  'far left to far right'),
]
tol = 1e-9
print(f"{'Description':<30}  {'Interval':<16}  {'Result':>10}  {'Expected':>10}  Status")
print('-' * 82)
for x_s, x_e, expected, desc in test_cases:
    v = float(interp_1d.integrate(x_s, x_e))
    status = 'PASS' if abs(v - expected) < tol else 'FAIL'
    interval = f"[{x_s}, {x_e}]"
    print(f"{desc:<30}  {interval:<16}  {v:>10.4f}  {expected:>10.4f}  [{status}]")

Description                     Interval              Result    Expected  Status
----------------------------------------------------------------------------------
both in left wing               [0.5, 0.9]            1.2000      1.2000  [PASS]
left wing to first bucket       [0.5, 1.2]            2.3000      2.3000  [PASS]
left wing to middle             [0.5, 3.2]           10.5000     10.5000  [PASS]
interior span                   [1.5, 5.2]           17.2000     17.2000  [PASS]
middle to right wing            [3.5, 7.2]           20.7000     20.7000  [PASS]
last bucket to right wing       [6, 7.2]              7.2000      7.2000  [PASS]
both in right wing              [8, 10]              12.0000     12.0000  [PASS]
full span                       [0.1, 10]            50.7000     50.7000  [PASS]
zero-width interval             [2, 2]                0.0000      0.0000  [PASS]
knot-to-knot [1,3)              [1, 3]                8.0000      8.0000  [PASS]
full interior [1,7)       

In [7]:
### 1D integrate gradient: analytic (backward called outside integrate()) vs bump-reval
tol = 1e-6
test_cases = [
    ([0.5, 0.9],  'both in left wing'),
    ([0.5, 1.2],  'left wing to first bucket'),
    ([0.5, 3.2],  'left wing to middle'),
    ([1.5, 5.2],  'interior span'),
    ([3.5, 7.2],  'middle to right wing'),
    ([6,   7.2],  'last bucket to right wing'),
    ([8,  10],    'both in right wing'),
    ([0.1, 10],   'full span'),
    ([2,   2],    'zero-width interval'),
    ([1,   3],    'knot-to-knot [1,3)'),
    ([1,   7],    'full interior [1,7)'),
]
print(f"{'Description':<30}  {'Interval':<16}  {'max|diff|':>12}  Status")
print('-' * 70)
for (x_s, x_e), desc in test_cases:
    grad_analytic = analytic_grad_of_integrated_value_wrt_ordinate_1d(x_s, x_e, axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    grad_br = bump_reval_interpolator_integrand(x_s, x_e, axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    max_diff = np.abs(grad_analytic - grad_br).max()
    status = 'PASS' if max_diff < tol else 'FAIL'
    interval = f"[{x_s}, {x_e}]"
    print(f"{desc:<30}  {interval:<16}  {max_diff:>12.2e}  [{status}]")

Description                     Interval             max|diff|  Status
----------------------------------------------------------------------
both in left wing               [0.5, 0.9]            4.00e-13  [PASS]
left wing to first bucket       [0.5, 1.2]            1.31e-12  [PASS]
left wing to middle             [0.5, 3.2]            1.66e-11  [PASS]
interior span                   [1.5, 5.2]            2.13e-11  [PASS]
middle to right wing            [3.5, 7.2]            2.13e-11  [PASS]
last bucket to right wing       [6, 7.2]              1.02e-12  [PASS]
both in right wing              [8, 10]               4.66e-12  [PASS]
full span                       [0.1, 10]             7.57e-11  [PASS]
zero-width interval             [2, 2]                0.00e+00  [PASS]
knot-to-knot [1,3)              [1, 3]                4.22e-12  [PASS]
full interior [1,7)             [1, 7]                4.66e-12  [PASS]


## Test 2D Interpolator -- Interpolate and Sensitivities

In [8]:
axis1_2d = [1, 3, 5, 7]
axis2_2d = [10, 20, 30]
values_2d = [[11, 12, 13],
             [21, 22, 23],
             [31, 32, 33],
             [41, 42, 43]]
interp_method_2d = 'LINEAR'
extrap_method_2d = 'FLAT'
interp_2d = qfCreate2DInterpolator(axis1_2d, axis2_2d, values_2d, interp_method_2d, extrap_method_2d)

# (x, y, expected, description) -- regular grid/interior points + corner cases (boundaries, extrapolation)
test_cases = [
    (1,    10,  11.0, 'grid corner (1,10)'),
    (1.5,  15,  14.0, 'interior point'),
    (3,    20,  22.0, 'grid point (3,20)'),
    (7,    30,  43.0, 'grid corner (7,30)'),
    (7,    10,  41.0, 'right x boundary, low y'),
    (1,    30,  13.0, 'left x, high y boundary'),
    (0.5,  15,  11.5, 'left x extrapolation'),
    (6.5,  15,  39.0, 'right x extrapolation'),
    (3,     5,  21.0, 'low y extrapolation'),
    (3,    35,  23.0, 'high y extrapolation'),
    (-10,  -5,  11.0, 'far left/low extrapolation'),
    (100, 100,  43.0, 'far right/high extrapolation'),
]
tol = 1e-10
print(f"{'Description':<35}  {'(x,y)':<16}  {'Result':>10}  {'Expected':>10}  Status")
print('-' * 86)
for x, y, expected, desc in test_cases:
    result = interp_2d.interpolate(x, y)
    status = 'PASS' if abs(result - expected) < tol else 'FAIL'
    point = f"({x}, {y})"
    print(f"{desc:<35}  {point:<16}  {result:>10.4f}  {expected:>10.4f}  [{status}]")

Description                          (x,y)                 Result    Expected  Status
--------------------------------------------------------------------------------------
grid corner (1,10)                   (1, 10)              11.0000     11.0000  [PASS]
interior point                       (1.5, 15)            14.0000     14.0000  [PASS]
grid point (3,20)                    (3, 20)              22.0000     22.0000  [PASS]
grid corner (7,30)                   (7, 30)              43.0000     43.0000  [PASS]
right x boundary, low y              (7, 10)              41.0000     41.0000  [PASS]
left x, high y boundary              (1, 30)              13.0000     13.0000  [PASS]
left x extrapolation                 (0.5, 15)            11.5000     11.5000  [PASS]
right x extrapolation                (6.5, 15)            39.0000     39.0000  [PASS]
low y extrapolation                  (3, 5)               21.0000     21.0000  [PASS]
high y extrapolation                 (3, 35)         

In [9]:
# utility function for 2D bump reval
def bump_reval_interpolator_2d(
    x : float,
    y : float,
    axis1_2d : List,
    axis2_2d : List,
    values_2d : List,
    interp_method_2d : str,
    extrap_method_2d : str,
    bump_size : Optional[float]=1e-4):

    base_interpolator = qfCreate2DInterpolator(axis1_2d, axis2_2d, values_2d, interp_method_2d, extrap_method_2d)
    b_value = base_interpolator.interpolate(x, y)

    grad = []
    for i in range(len(values_2d)):
        for j in range(len(values_2d[0])):
            values_2d[i][j] += bump_size
            this_interp = qfCreate2DInterpolator(axis1_2d, axis2_2d, values_2d, interp_method_2d, extrap_method_2d)
            bumped_value = this_interp.interpolate(x, y)
            grad.append((bumped_value - b_value) / bump_size)
            values_2d[i][j] -= bump_size

    return np.array(grad)

In [10]:
### 2D interpolate gradient: analytic (autograd) vs bump-reval
# Interpolator2D still uses the calc_grad / convert_to_numpy + gradient_wrt_ordinate API.
tol = 1e-6
test_cases = [
    ((1,   10),  'grid corner (1,10)'),
    ((1.5, 15),  'interior point'),
    ((3,   20),  'grid point (3,20)'),
    ((7,   30),  'grid corner (7,30)'),
    ((7,   10),  'right x boundary, low y'),
    ((1,   30),  'left x, high y boundary'),
    ((0.5, 15),  'left x extrapolation'),
    ((6.5, 15),  'right x extrapolation'),
    ((3,    5),  'low y extrapolation'),
    ((3,   35),  'high y extrapolation'),
    ((-10, -5),  'far left/low extrapolation'),
    ((100,100),  'far right/high extrapolation'),
]
print(f"{'Description':<35}  {'(x,y)':<16}  {'max|diff|':>12}  Status")
print('-' * 76)
for (x, y), desc in test_cases:
    grad_analytic = interp_2d.gradient_wrt_ordinate(x, y, convert_to_numpy=True).flatten()
    grad_br = bump_reval_interpolator_2d(x, y, axis1_2d, axis2_2d, values_2d, interp_method_2d, extrap_method_2d)
    max_diff = np.abs(grad_analytic - grad_br).max()
    status = 'PASS' if max_diff < tol else 'FAIL'
    point = f"({x}, {y})"
    print(f"{desc:<35}  {point:<16}  {max_diff:>12.2e}  [{status}]")

Description                          (x,y)                max|diff|  Status
----------------------------------------------------------------------------
grid corner (1,10)                   (1, 10)               2.33e-12  [PASS]
interior point                       (1.5, 15)             5.31e-12  [PASS]
grid point (3,20)                    (3, 20)               2.33e-12  [PASS]
grid corner (7,30)                   (7, 30)               3.32e-11  [PASS]
right x boundary, low y              (7, 10)               3.32e-11  [PASS]
left x, high y boundary              (1, 30)               2.33e-12  [PASS]
left x extrapolation                 (0.5, 15)             1.17e-12  [PASS]
right x extrapolation                (6.5, 15)             4.80e-11  [PASS]
low y extrapolation                  (3, 5)                2.33e-12  [PASS]
high y extrapolation                 (3, 35)               2.33e-12  [PASS]
far left/low extrapolation           (-10, -5)             2.33e-12  [PASS]
far right/h

## Vectorized (Array) Input Corner Cases

These cells verify that array inputs produce the same results as the corresponding scalar calls.
For 1D, `interpolate`/`integrate` accept array-like `x` directly (no manual array wrapping needed).
For 1D gradients, calling `interpolate`/`integrate` with `calc_grad=True` on an array and calling
`.backward()` on the **sum** of the result, then reading `interp.values_.grad`, gives the **sum**
of per-point gradients (grad of `sum_i f(x_i)` w.r.t. values) -- the analogue of the old
`gradient_wrt_ordinate` on an array. 2D keeps its own array-capable `interpolate`/`gradient_wrt_ordinate`.

In [11]:
### Vectorized corner cases — 1D interpolate & integrate
# Re-uses interp_1d (axis=[1,3,5,7], values=[3,4,5,6]) from cell above

def _scalar_batch(fn, *arg_arrs):
    """Call fn with scalar args for each position and stack into an array."""
    return np.array([to_np(fn(*[float(a[i]) for a in arg_arrs])) for i in range(len(arg_arrs[0]))])

tol_v = 1e-12   # value tolerance
tol_g = 1e-10   # gradient tolerance

# ── 1D interpolate ────────────────────────────────────────────────────────────
print("=" * 66)
print("1D interpolate — array input")
print("=" * 66)
print(f"{'Description':<30}  {'Shape':>7}  {'Max|diff|':>12}  Status")
print('-' * 60)

cases_1d_interp = [
    ('single-element array',   np.array([1.5])),
    ('all left extrap',        np.array([-100.0, -1, 0.5, 0.99])),
    ('all right extrap',       np.array([7.0, 8, 100, 999])),
    ('all at knots',           np.array([1.0, 3, 5, 7])),
    ('mixed interior+extrap',  np.array([-5.0, 1, 1.5, 3, 4, 5.5, 7, 20])),
    ('duplicate x values',     np.array([3.0, 3, 3, 3])),
]
for desc, x_arr in cases_1d_interp:
    res = to_np(interp_1d.interpolate(x_arr))
    exp = _scalar_batch(interp_1d.interpolate, x_arr)
    diff = np.abs(res - exp).max()
    s = 'PASS' if diff < tol_v else 'FAIL'
    print(f"{desc:<30}  {str(res.shape):>7}  {diff:>12.2e}  [{s}]")

# ── 1D integrate ─────────────────────────────────────────────────────────────
print()
print("=" * 66)
print("1D integrate — array input")
print("=" * 66)
print(f"{'Description':<30}  {'Shape':>7}  {'Max|diff|':>12}  Status")
print('-' * 60)

cases_1d_integ = [
    ('single pair',           np.array([1.5]),           np.array([4.5])),
    ('all zero-width',        np.array([1.0, 3, 5]),     np.array([1.0, 3, 5])),
    ('all left extrap',       np.array([-5.0, -3]),      np.array([-2.0, -1])),
    ('all right extrap',      np.array([8.0, 10]),       np.array([9.0, 12])),
    ('reversed s > e → 0',   np.array([3.0, 5]),        np.array([1.0, 2])),
    ('mixed spans',           np.array([-5.0, 1, 3]),    np.array([1.0, 5, 15])),
    ('same start, diff end',  np.array([1.0, 1, 1]),     np.array([3.0, 5, 7])),
    ('same end, diff start',  np.array([1.0, 3, 5]),     np.array([7.0, 7, 7])),
]
for desc, s_arr, e_arr in cases_1d_integ:
    res = to_np(interp_1d.integrate(s_arr, e_arr))
    exp = _scalar_batch(interp_1d.integrate, s_arr, e_arr)
    diff = np.abs(res - exp).max()
    s = 'PASS' if diff < tol_v else 'FAIL'
    print(f"{desc:<30}  {str(res.shape):>7}  {diff:>12.2e}  [{s}]")

1D interpolate — array input
Description                       Shape     Max|diff|  Status
------------------------------------------------------------
single-element array               (1,)      0.00e+00  [PASS]
all left extrap                    (4,)      0.00e+00  [PASS]
all right extrap                   (4,)      0.00e+00  [PASS]
all at knots                       (4,)      0.00e+00  [PASS]
mixed interior+extrap              (8,)      0.00e+00  [PASS]
duplicate x values                 (4,)      0.00e+00  [PASS]

1D integrate — array input
Description                       Shape     Max|diff|  Status
------------------------------------------------------------
single pair                        (1,)      0.00e+00  [PASS]
all zero-width                     (3,)      0.00e+00  [PASS]
all left extrap                    (2,)      0.00e+00  [PASS]
all right extrap                   (2,)      0.00e+00  [PASS]
reversed s > e → 0                 (2,)      0.00e+00  [PASS]
mixed spans    

In [12]:
### Vectorized corner cases — 1D gradients
# Calling interpolate/integrate(array, calc_grad=True), summing the result, and calling
# .backward() outside gives the SUM of per-point gradients (grad of sum_i f(x_i) wrt values),
# shape (n_values,). This matches the old gradient_wrt_ordinate(array) semantics.

tol_g = 1e-10

# ── interpolate gradient ──────────────────────────────────────────────────────
print("=" * 68)
print("1D interpolate gradient — array x  (returns summed grad, shape (n,))")
print("=" * 68)
print(f"{'Description':<28}  {'grad shape':>12}  {'Max|diff|':>12}  Status")
print('-' * 64)

cases_1d_grad_interp = [
    ('single element',    np.array([1.5])),
    ('all same point',    np.array([3.0, 3.0, 3.0])),   # grad = 3 × one-hot
    ('all left extrap',   np.array([-5.0, -1, 0.5])),
    ('all right extrap',  np.array([7.5, 10.0, 100])),
    ('knot points only',  np.array([1.0, 3, 5, 7])),
    ('mixed all regions', np.array([-1.0, 1, 2, 3, 4, 5, 6, 7, 8])),
]
for desc, x_arr in cases_1d_grad_interp:
    g_vec = analytic_grad_wrt_ordinate_1d(x_arr, axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    g_exp = sum(analytic_grad_wrt_ordinate_1d(float(xi), axis1_1d, values_1d, interp_method_1d, extrap_method_1d) for xi in x_arr)
    diff = np.abs(g_vec - g_exp).max()
    s = 'PASS' if diff < tol_g else 'FAIL'
    print(f"{desc:<28}  {str(g_vec.shape):>12}  {diff:>12.2e}  [{s}]")

# ── integrate gradient ────────────────────────────────────────────────────────
print()
print("=" * 68)
print("1D integrate gradient — array (s,e)  (summed grad, shape (n,))")
print("=" * 68)
print(f"{'Description':<28}  {'grad shape':>12}  {'Max|diff|':>12}  Status")
print('-' * 64)

cases_1d_grad_integ = [
    ('single pair',        np.array([1.5]),        np.array([4.5])),
    ('all zero-width',     np.array([1.0, 3, 5]),  np.array([1.0, 3, 5])),  # grad = 0
    ('all same interval',  np.array([1.0, 1, 1]),  np.array([3.0, 3, 3])),  # grad = 3×single
    ('mixed spans',        np.array([-5.0, 1, 3]), np.array([1.0, 5, 15])),
    ('extrap intervals',   np.array([-10.0, 8]),   np.array([-5.0, 12])),
]
for desc, s_arr, e_arr in cases_1d_grad_integ:
    g_vec = analytic_grad_of_integrated_value_wrt_ordinate_1d(s_arr, e_arr, axis1_1d, values_1d, interp_method_1d, extrap_method_1d)
    g_exp = sum(analytic_grad_of_integrated_value_wrt_ordinate_1d(
        float(s), float(e), axis1_1d, values_1d, interp_method_1d, extrap_method_1d) for s, e in zip(s_arr, e_arr))
    diff = np.abs(g_vec - g_exp).max()
    s = 'PASS' if diff < tol_g else 'FAIL'
    print(f"{desc:<28}  {str(g_vec.shape):>12}  {diff:>12.2e}  [{s}]")

1D interpolate gradient — array x  (returns summed grad, shape (n,))
Description                     grad shape     Max|diff|  Status
----------------------------------------------------------------
single element                        (4,)      0.00e+00  [PASS]
all same point                        (4,)      0.00e+00  [PASS]
all left extrap                       (4,)      0.00e+00  [PASS]
all right extrap                      (4,)      0.00e+00  [PASS]
knot points only                      (4,)      0.00e+00  [PASS]
mixed all regions                     (4,)      0.00e+00  [PASS]

1D integrate gradient — array (s,e)  (summed grad, shape (n,))
Description                     grad shape     Max|diff|  Status
----------------------------------------------------------------
single pair                           (4,)      0.00e+00  [PASS]
all zero-width                        (4,)      0.00e+00  [PASS]
all same interval                     (4,)      0.00e+00  [PASS]
mixed spans           

In [13]:
### Vectorized corner cases — 2D interpolate & gradient
# Re-uses interp_2d (axis1_2d=[1,3,5,7], axis2_2d=[10,20,30]) from cell above

tol_v = 1e-12
tol_g = 1e-10

cases_2d = [
    ('single pair',           np.array([1.5]),               np.array([15.0])),
    ('all grid corners',      np.array([1, 1, 7, 7.0]),      np.array([10, 30, 10, 30.0])),
    ('all left-x extrap',     np.array([-5, -5, -5.0]),      np.array([10, 20, 30.0])),
    ('all high-y extrap',     np.array([2, 4, 6.0]),         np.array([50, 50, 50.0])),
    ('both axes extrap',      np.array([-10, 100.0]),        np.array([-5, 100.0])),
    ('same x, varying y',     np.array([3, 3, 3.0]),         np.array([10, 20, 30.0])),
    ('varying x, same y',     np.array([1, 3, 5, 7.0]),     np.array([20, 20, 20, 20.0])),
    ('mixed interior+extrap', np.array([0.5, 2, 4, 7.5]),   np.array([5, 15, 25, 35.0])),
]

# ── 2D interpolate ────────────────────────────────────────────────────────────
print("=" * 66)
print("2D interpolate — array input")
print("=" * 66)
print(f"{'Description':<28}  {'Shape':>7}  {'Max|diff|':>12}  Status")
print('-' * 60)

for desc, x_arr, y_arr in cases_2d:
    res = interp_2d.interpolate(x_arr, y_arr)
    exp = _scalar_batch(interp_2d.interpolate, x_arr, y_arr)
    diff = np.abs(res - exp).max()
    s = 'PASS' if diff < tol_v else 'FAIL'
    print(f"{desc:<28}  {str(res.shape):>7}  {diff:>12.2e}  [{s}]")

# ── 2D gradient_wrt_ordinate ──────────────────────────────────────────────────
# Returns shape (size1, size2) = sum of per-point bilinear-weight matrices.
print()
print("=" * 68)
print("2D gradient_wrt_ordinate — array input  (summed grad, shape (s1,s2))")
print("=" * 68)
print(f"{'Description':<28}  {'grad shape':>12}  {'Max|diff|':>12}  Status")
print('-' * 64)

for desc, x_arr, y_arr in cases_2d:
    g_vec = interp_2d.gradient_wrt_ordinate(x_arr, y_arr, convert_to_numpy=True)
    g_exp = sum(
        interp_2d.gradient_wrt_ordinate(float(x), float(y), convert_to_numpy=True)
        for x, y in zip(x_arr, y_arr)
    )
    diff = np.abs(g_vec - g_exp).max()
    s = 'PASS' if diff < tol_g else 'FAIL'
    print(f"{desc:<28}  {str(g_vec.shape):>12}  {diff:>12.2e}  [{s}]")

2D interpolate — array input
Description                     Shape     Max|diff|  Status
------------------------------------------------------------
single pair                      (1,)      0.00e+00  [PASS]
all grid corners                 (4,)      0.00e+00  [PASS]
all left-x extrap                (3,)      0.00e+00  [PASS]
all high-y extrap                (3,)      0.00e+00  [PASS]
both axes extrap                 (2,)      0.00e+00  [PASS]
same x, varying y                (3,)      0.00e+00  [PASS]
varying x, same y                (4,)      0.00e+00  [PASS]
mixed interior+extrap            (4,)      0.00e+00  [PASS]

2D gradient_wrt_ordinate — array input  (summed grad, shape (s1,s2))
Description                     grad shape     Max|diff|  Status
----------------------------------------------------------------
single pair                         (4, 3)      0.00e+00  [PASS]
all grid corners                    (4, 3)      0.00e+00  [PASS]
all left-x extrap                   (4, 